# 10 — Pricing for profit: elastic demand vs arbitrage

Being un-arbitrageable is not the goal; **making the most money is**. Profit is house edge times
volume, and volume depends on how competitive the odds are. Wider margins (to price out
arbitrageurs) also drive flow away. Because the net-delta skew caps one-sided exposure, a small
arbitrage leak can be far cheaper than the volume lost by over-charging.

To test this we make demand **elastic**: `demand_responsive_pool` scales its volume with the
offered margin (better odds -> more volume). We then sweep the house margin (the competitiveness
dial) on the `volatility_regime_momentum` model, with the realistic **regime-aware arbitrageur**
betting inside the pool, and read off the **net house PnL**.

Assumptions: total traded volume is calibrated to ~200,000 at the base vig over the evaluation
window; demand elasticity is 8.0 (sensitivity matters — see the note at the end).

In [ ]:
import os
import sys

# Move up one level to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# Add the new current directory to the Python path
sys.path.insert(0, os.getcwd())

In [ ]:
import dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snapmarket.data import load_oracle_prices
from snapmarket.features import build_features
from snapmarket.parameters import SharedParameters
from snapmarket.models import build_model
from snapmarket.signals import regime_conditional_probability
from snapmarket.strategies import demand_responsive_pool, regime_aware_bettor
from snapmarket.engine import simulate

shared_parameters = SharedParameters()
features = build_features(load_oracle_prices(), shared_parameters)
base_model = build_model("volatility_regime_momentum", features, shared_parameters)
regime_probability = regime_conditional_probability(features, shared_parameters)

evaluation_start = base_model.first_evaluation_index
window_length = 400_000
elasticity = 8.0
reference_margin = 0.125
target_volume = 200_000

def model_with_margin(margin):
    return dataclasses.replace(base_model, margin=float(margin))

## Calibrate the demand level

Set the pool's base stake so that, at the base vig, the pool trades ~`target_volume` over the
window. Net PnL scales linearly with this, so the shape of the curve is what matters.

In [ ]:
reference = simulate(model_with_margin(reference_margin), features,
                     {"pool": demand_responsive_pool(1.0, elasticity, reference_margin)},
                     evaluation_start, window_length, seed=7)
base_stake = target_volume / reference.per_bettor["pool"].stake
print(f"calibrated base stake = {base_stake:.4f}")

## Sweep the house margin

For each margin: the elastic pool and the regime-aware arbitrageur bet together; we record total
volume, the arbitrageur's loss to the house, and the net house PnL.

In [ ]:
margins = [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25]

rows = []
for margin in margins:
    result = simulate(
        model_with_margin(margin), features,
        {"pool": demand_responsive_pool(base_stake, elasticity, reference_margin),
         "attacker": regime_aware_bettor(regime_probability, base_stake=base_stake)},
        evaluation_start, window_length, seed=7)
    attacker = result.per_bettor["attacker"]
    rows.append({"margin": margin, "total_volume": result.total_volume,
                 "attacker_edge": attacker.edge, "arbitrage_loss_to_house": -attacker.pnl,
                 "net_house_pnl": result.house_pnl, "net_house_edge": result.house_edge})

sweep = pd.DataFrame(rows).set_index("margin")
print(sweep.round(2).to_string())
optimum = sweep["net_house_pnl"].idxmax()
print(f"\nProfit-maximising margin = {optimum:.3f}  (net house PnL {sweep.loc[optimum, 'net_house_pnl']:,.0f})")
print(f"Arbitrage-proof margin 0.25 net PnL = {sweep.loc[0.25, 'net_house_pnl']:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(sweep.index, sweep["net_house_pnl"], "o-", color="#2563eb")
axes[0].axvline(optimum, ls=":", color="#16a34a", label=f"optimum margin = {optimum:.3f}")
axes[0].set_title("Net house PnL vs margin (elastic demand + arbitrageur)")
axes[0].set_xlabel("house margin"); axes[0].set_ylabel("net house PnL")
axes[0].legend(fontsize=8)

axes[1].plot(sweep.index, sweep["total_volume"], "o-", color="#9ca3af", label="total volume")
axes[1].plot(sweep.index, sweep["arbitrage_loss_to_house"], "o-", color="#dc2626",
             label="arbitrage loss to house")
axes[1].set_title("Volume and arbitrage loss vs margin")
axes[1].set_xlabel("house margin"); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Recorded results

Evaluation window of 400,000 seconds from the common out-of-sample start; demand elasticity 8.0;
total volume calibrated to ~200,000 at the 0.125 vig.

| house margin | total volume | arbitrage loss to house | net house PnL |
|---|---|---|---|
| 0.050 | 483,653 | -4,339 | 16,124 |
| 0.075 | 369,082 | -1,968 | 22,180 |
| 0.100 | 277,660 | -1,133 | 24,692 |
| 0.125 | 221,227 | -845 | **25,256** |
| 0.150 | 174,537 | -327 | 25,102 |
| 0.200 | 113,104 | -31 | 22,456 |
| 0.250 | 73,507 | 0 | 18,721 |

**Read.** Net house PnL is an inverted U with an interior optimum near a 0.125-0.15 margin. The
**arbitrage-proof** end (0.25, where the arbitrageur makes nothing) earns only ~18,700 — about
**26% less** than the optimum (~25,300), purely because it scares off volume. At the optimum the
house does leak a little to the arbitrageur (-845), but that loss is tiny next to the profit the
extra volume brings in. Over-competing (0.05) is also worse: the edge per unit shrinks and
arbitrage grows faster than volume.

So the answer to the original question is **yes**: being slightly arbitrageable is not a problem,
it is optimal — price for the profit-maximising margin, not for zero arbitrage. The net-delta skew
keeps the arbitrage loss small and bounded, exactly as hoped.

**Caveat.** The exact optimum depends on the demand elasticity (assumed 8.0) and the volume scale.
A more elastic market pushes the optimum to tighter margins; a less elastic one to wider margins.
The qualitative shape — an interior optimum below the arbitrage-proof margin — is the robust result.

![net house PnL vs margin](assets/10_pricing_for_profit.png)